In [ ]:
import json

import pandas as pd
import great_expectations as gx
import synapseclient

from agoradatatools.gx import GreatExpectationsRunner

context = gx.get_context(project_root_dir='../src/agoradatatools/great_expectations')

from expectations.expect_column_values_to_have_list_members import ExpectColumnValuesToHaveListMembers
from expectations.expect_column_values_to_have_list_members_of_type import ExpectColumnValuesToHaveListMembersOfType
from expectations.expect_column_values_to_have_list_length_in_range import ExpectColumnValuesToHaveListLengthInRange

# Create Expectation Suite for Model Overview Data

## Get Example Data File

In [ ]:
syn = synapseclient.Synapse()
syn.login()

In [ ]:
# Replace <SYN_ID> with the Synapse ID of the processed model_overview output file.
# Alternatively, use the local staging file: model_overview_file = "../staging/model_overview.json"
model_overview_file = syn.get("syn73774526").path

## Create Validator Object on Data File

In [ ]:
df = pd.read_json(model_overview_file)
nested_columns = [
    "transcriptomics",
    "disease_correlation",
    "pathology",
    "biomarkers",
    "study_data",
    "jax_strain",
    "center",
]
df = GreatExpectationsRunner.convert_nested_columns_to_json(df, nested_columns)
validator = context.sources.pandas_default.read_dataframe(df)
validator.expectation_suite_name = "model_overview"

## Add Expectations to Validator Object For Each Column

In [ ]:
# name
validator.expect_column_values_to_be_of_type("name", "str")
validator.expect_column_values_to_not_be_null("name")
validator.expect_column_values_to_be_unique("name")

In [ ]:
# model_type
validator.expect_column_values_to_be_of_type("model_type", "str")
validator.expect_column_values_to_not_be_null("model_type")
validator.expect_column_values_to_be_in_set("model_type", {"Familial AD", "Late Onset AD"})

In [ ]:
# matched_controls
validator.expect_column_values_to_be_of_type("matched_controls", "list")
validator.expect_column_values_to_not_be_null("matched_controls")
validator.expect_column_values_to_have_list_length_in_range(column="matched_controls", list_length_range=[1, 2])
validator.expect_column_values_to_have_list_members_of_type(column="matched_controls", member_type="str")

In [ ]:
# modified_genes
validator.expect_column_values_to_be_of_type("modified_genes", "list")
validator.expect_column_values_to_not_be_null("modified_genes")
validator.expect_column_values_to_have_list_length_in_range(column="modified_genes", list_length_range=[1, 10])
validator.expect_column_values_to_have_list_members_of_type(column="modified_genes", member_type="str")

In [ ]:
# available_data
validator.expect_column_values_to_be_of_type("available_data", "list")
validator.expect_column_values_to_not_be_null("available_data")
validator.expect_column_values_to_have_list_length_in_range(column="available_data", list_length_range=[1, 4])
validator.expect_column_values_to_have_list_members_of_type(column="available_data", member_type="str")
validator.expect_column_values_to_have_list_members(
    column="available_data",
    list_members={"Transcriptomics", "Disease Correlation", "Pathology", "Biomarkers"}
)

In [ ]:
# transcriptomics, disease_correlation, pathology, biomarkers (nullable link columns)
# Each schema validates the nested link_url against the expected URL pattern for that column.
with open("../src/agoradatatools/great_expectations/gx/json_schemas/model_overview/transcriptomics_schema.json", "r") as file:
    transcriptomics_schema = json.load(file)

with open("../src/agoradatatools/great_expectations/gx/json_schemas/model_overview/disease_correlation_schema.json", "r") as file:
    disease_correlation_schema = json.load(file)

with open("../src/agoradatatools/great_expectations/gx/json_schemas/model_overview/pathology_schema.json", "r") as file:
    pathology_schema = json.load(file)

with open("../src/agoradatatools/great_expectations/gx/json_schemas/model_overview/biomarkers_schema.json", "r") as file:
    biomarkers_schema = json.load(file)

validator.expect_column_values_to_be_of_type("transcriptomics", "str")
validator.expect_column_values_to_match_json_schema("transcriptomics", json_schema=transcriptomics_schema)

validator.expect_column_values_to_be_of_type("disease_correlation", "str")
validator.expect_column_values_to_match_json_schema("disease_correlation", json_schema=disease_correlation_schema)

validator.expect_column_values_to_be_of_type("pathology", "str")
validator.expect_column_values_to_match_json_schema("pathology", json_schema=pathology_schema)

validator.expect_column_values_to_be_of_type("biomarkers", "str")
validator.expect_column_values_to_match_json_schema("biomarkers", json_schema=biomarkers_schema)

In [ ]:
# study_data, jax_strain (required link columns - must never be null)
with open("../src/agoradatatools/great_expectations/gx/json_schemas/model_overview/link_url_required_schema.json", "r") as file:
    link_url_required_schema = json.load(file)

validator.expect_column_values_to_be_of_type("study_data", "str")
validator.expect_column_values_to_match_json_schema("study_data", json_schema=link_url_required_schema)

validator.expect_column_values_to_be_of_type("jax_strain", "str")
validator.expect_column_values_to_match_json_schema("jax_strain", json_schema=link_url_required_schema)

In [ ]:
# center (required - must never be null, has both link_url and link_text)
with open("../src/agoradatatools/great_expectations/gx/json_schemas/model_overview/center_schema.json", "r") as file:
    center_schema = json.load(file)

validator.expect_column_values_to_be_of_type("center", "str")
validator.expect_column_values_to_match_json_schema("center", json_schema=center_schema)

## Save Expectation Suite

In [ ]:
validator.save_expectation_suite(discard_failed_expectations=False)

## Create Checkpoint and View Results

In [ ]:
checkpoint = context.add_or_update_checkpoint(
    name="agora-test-checkpoint",
    validator=validator,
)
checkpoint_result = checkpoint.run()
context.view_validation_result(checkpoint_result)

## Build Data Docs - Click on Expectation Suite to View All Expectations

In [ ]:
context.build_data_docs()
context.open_data_docs()